# Tutorial: Chunked Agent Analysis of Sentiment Labelled Sentences

Audience:
- Engineers evaluating `pyflow` as a notebook-first framework for chunked agent workflows.

Prerequisites:
- Run the notebook from this repository with the local `.venv` kernel.
- Set `GROQ_API_KEY` in the environment or in a local `.env` file.
- Basic familiarity with `pandas` and Python functions.

Learning goals:
- Load and normalize a small text dataset.
- Split the dataset into deterministic chunks.
- Process chunks with `Agent.parallel(...)`.
- Merge structured agent outputs with plain Python.
- Run one reducer pass that turns merged tables into a final narrative.


## Outline

1. Setup the dataset and agent.
2. Load and normalize the review sentences.
3. Split the rows into deterministic chunks.
4. Analyze one chunk interactively.
5. Run the full chunk set with `Agent.parallel(...)`.
6. Merge the structured outputs with `pandas`.
7. Ask one reducer agent for the final narrative.
8. Try one extension.


In [17]:
import json
import sys
from pathlib import Path

import pandas as pd
from pydantic import BaseModel

ROOT = Path("..")
DATASET_DIR = ROOT / "sentiment-labelled-sentences"
SOURCE_FRACTION = 0.25
CHUNK_SIZE = 25
MAX_CONCURRENCY = 6

sys.path.append(str(ROOT))

from pyflow import ParallelFailure, Request, output
from model_setup import empty_agent as agent

pd.set_option("display.max_colwidth", 120)

from utils import ChunkSpec, build_chunks, load_sentiment_dataset


## Step 1 - Load and inspect the dataset

This dataset ships as three tab-separated files. To keep the demo fast, the notebook keeps only the first quarter of rows from each source, preserving a balanced subset with stable `row_id`, `source`, and label columns.


In [18]:
full_dataset = load_sentiment_dataset(DATASET_DIR)
rows_per_source = max(
    1,
    int(full_dataset.groupby("source").size().min() * SOURCE_FRACTION),
)
dataset = (
    full_dataset.groupby("source", group_keys=False)
    .head(rows_per_source)
    .reset_index(drop=True)
)

dataset.head(10)


,row_id,source,label,label_name,text
0,0,amazon,0,negative,So there is no way for me to plug it in here in the US unless I go by a converter.
1,1,amazon,1,positive,"Good case, Excellent value."
2,2,amazon,1,positive,Great for the jawbone.
3,3,amazon,0,negative,Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!
4,4,amazon,1,positive,The mic is great.
5,5,amazon,0,negative,I have to jiggle the plug to get it to line up right to get decent volume.
6,6,amazon,0,negative,"If you have several dozen or several hundred contacts, then imagine the fun of sending each of them one by one."
7,7,amazon,1,positive,If you are Razr owner...you must have this!
8,8,amazon,0,negative,"Needless to say, I wasted my money."
9,9,amazon,0,negative,What a waste of money and time!.


In [19]:
dataset.groupby(["source", "label_name"]).size().unstack(fill_value=0)

label_name,negative,positive
source,,
amazon,123,127
imdb,151,99
yelp,114,136


## Step 2 - Split the dataset into deterministic chunks

Each chunk becomes one independent request. The split is code, not AI: group by source, preserve row order, then take fixed 25-row windows so the process stays repeatable and fast enough for a demo.


In [20]:
chunks = build_chunks(dataset, CHUNK_SIZE)

chunk_overview = pd.DataFrame(
    [
        {
            "source": chunk.source,
            "chunk_index": chunk.chunk_index,
            "row_count": chunk.row_count,
            "positive_rows": int((chunk.rows["label"] == 1).sum()),
            "negative_rows": int((chunk.rows["label"] == 0).sum()),
        }
        for chunk in chunks
    ]
)

chunk_overview

,source,chunk_index,row_count,positive_rows,negative_rows
0,amazon,0,25,12,13
1,amazon,1,25,11,14
2,amazon,2,25,15,10
3,amazon,3,25,13,12
4,amazon,4,25,17,8
5,amazon,5,25,10,15
6,amazon,6,25,14,11
7,amazon,7,25,9,16
8,amazon,8,25,14,11
9,amazon,9,25,12,13


In [21]:
chunks[0].rows.head(5)

,row_id,source,label,label_name,text
0,0,amazon,0,negative,So there is no way for me to plug it in here in the US unless I go by a converter.
1,1,amazon,1,positive,"Good case, Excellent value."
2,2,amazon,1,positive,Great for the jawbone.
3,3,amazon,0,negative,Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!
4,4,amazon,1,positive,The mic is great.


## Step 3 - Define the chunk analysis contract

The agent returns one strict object per chunk. The notebook declares that shape as Pydantic models, attaches it with `// output(...)`, and lets `session.result` do the parsing and validation.


In [22]:
@dataclass(frozen=True)
class ChunkSpec:
    source: str
    chunk_index: int
    rows: pd.DataFrame

    @property
    def row_count(self) -> int:
        return len(self.rows)

class ThemeSummary(BaseModel):
    theme: str
    count: int
    example_row_ids: list[int]

class ChunkAnalysis(BaseModel):
    source: str
    chunk_index: int
    row_count: int
    positive_themes: list[ThemeSummary]
    negative_themes: list[ThemeSummary]
    notable_contrasts: list[str]
    quality_notes: list[str]

def chunk_records_json(chunk: ChunkSpec) -> str:
    records = chunk.rows[["row_id", "label_name", "text"]].to_dict(orient="records")
    return json.dumps(records, ensure_ascii=False, indent=2)

def build_chunk_request(chunk: ChunkSpec) -> Request:
    return f"""
Analyze one chunk of a labeled sentiment dataset.

Chunk metadata:
- source: {chunk.source}
- chunk_index: {chunk.chunk_index}
- row_count: {chunk.row_count}

Rules:
- Use only the rows in this chunk.
- Copy `source`, `chunk_index`, and `row_count` from the chunk metadata.
- Keep theme names short and normalized.
- Include at most 5 themes for each sentiment.
- Counts must refer to rows in this chunk.
- Example row ids must exist in this chunk.

Rows:
{chunk_records_json(chunk)}
""".strip() // output(ChunkAnalysis)


print(build_chunk_request(chunks[0]).render()) #[:1400]

1. Analyze one chunk of a labeled sentiment dataset.
   
   Chunk metadata:
   - source: amazon
   - chunk_index: 0
   - row_count: 25
   
   Rules:
   - Use only the rows in this chunk.
   - Copy `source`, `chunk_index`, and `row_count` from the chunk metadata.
   - Keep theme names short and normalized.
   - Include at most 5 themes for each sentiment.
   - Counts must refer to rows in this chunk.
   - Example row ids must exist in this chunk.
   
   Rows:
   [
     {
       "row_id": 0,
       "label_name": "negative",
       "text": "So there is no way for me to plug it in here in the US unless I go by a converter."
     },
     {
       "row_id": 1,
       "label_name": "positive",
       "text": "Good case, Excellent value."
     },
     {
       "row_id": 2,
       "label_name": "positive",
       "text": "Great for the jawbone."
     },
     {
       "row_id": 3,
       "label_name": "negative",
       "text": "Tied to charger for conversations lasting more than 45 minutes.MAJO

## Step 4 - Run one chunk interactively

Start with one chunk so you can inspect the rendered request and the parsed `ChunkAnalysis` object before scaling out.


In [23]:
preview_chunk = chunks[0]
preview_session = build_chunk_request(preview_chunk) >> agent
preview_session


Status: **Finished**

2 turns · 1 tool call

### User

1. Analyze one chunk of a labeled sentiment dataset.
   
   Chunk metadata:
   - source: amazon
   - chunk_index: 0
   - row_count: 25
   
   Rules:
   - Use only the rows in this chunk.
   - Copy `source`, `chunk_index`, and `row_count` from the chunk metadata.
   - Keep theme names short and normalized.
   - Include at most 5 themes for each sentiment.
   - Counts must refer to rows in this chunk.
   - Example row ids must exist in this chunk.
   
   Rows:
   [
     {
       "row_id": 0,
       "label_name": "negative",
       "text": "So there is no way for me to plug it in here in the US unless I go by a converter."
     },
     {
       "row_id": 1,
       "label_name": "positive",
       "text": "Good case, Excellent value."
     },
     {
       "row_id": 2,
       "label_name": "positive",
       "text": "Great for the jawbone."
     },
     {
       "row_id": 3,
       "label_name": "negative",
       "text": "Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!"
     },
     {
       "row_id": 4,
       "label_name": "positive",
       "text": "The mic is great."
     },
     {
       "row_id": 5,
       "label_name": "negative",
       "text": "I have to jiggle the plug to get it to line up right to get decent volume."
     },
     {
       "row_id": 6,
       "label_name": "negative",
       "text": "If you have several dozen or several hundred contacts, then imagine the fun of sending each of them one by one."
     },
     {
       "row_id": 7,
       "label_name": "positive",
       "text": "If you are Razr owner...you must have this!"
     },
     {
       "row_id": 8,
       "label_name": "negative",
       "text": "Needless to say, I wasted my money."
     },
     {
       "row_id": 9,
       "label_name": "negative",
       "text": "What a waste of money and time!."
     },
     {
       "row_id": 10,
       "label_name": "positive",
       "text": "And the sound quality is great."
     },
     {
       "row_id": 11,
       "label_name": "positive",
       "text": "He was very impressed when going from the original battery to the extended battery."
     },
     {
       "row_id": 12,
       "label_name": "negative",
       "text": "If the two were seperated by a mere 5+ ft I started to notice excessive static and garbled sound from the headset."
     },
     {
       "row_id": 13,
       "label_name": "positive",
       "text": "Very good quality though"
     },
     {
       "row_id": 14,
       "label_name": "negative",
       "text": "The design is very odd, as the ear \"clip\" is not very comfortable at all."
     },
     {
       "row_id": 15,
       "label_name": "positive",
       "text": "Highly recommend for any one who has a blue tooth phone."
     },
     {
       "row_id": 16,
       "label_name": "negative",
       "text": "I advise EVERYONE DO NOT BE FOOLED!"
     },
     {
       "row_id": 17,
       "label_name": "positive",
       "text": "So Far So Good!."
     },
     {
       "row_id": 18,
       "label_name": "positive",
       "text": "Works great!."
     },
     {
       "row_id": 19,
       "label_name": "negative",
       "text": "It clicks into place in a way that makes you wonder how long that mechanism would last."
     },
     {
       "row_id": 20,
       "label_name": "negative",
       "text": "I went on Motorola's website and followed all directions, but could not get it to pair again."
     },
     {
       "row_id": 21,
       "label_name": "positive",
       "text": "I bought this to use with my Kindle Fire and absolutely loved it!"
     },
     {
       "row_id": 22,
       "label_name": "negative",
       "text": "The commercials are the most misleading."
     },
     {
       "row_id": 23,
       "label_name": "positive",
       "text": "I have yet to run this new battery below two bars and that's three days without charging."
     },
     {
       "row_id": 24,
       "label_name": "negative",
       "text": "I bought it for my mother and she had a problem with the battery."
     }
   ]

Output contract:
Return the final answer by calling `finish`.
Set `finish.message` to raw JSON that matches this schema exactly.
Do not wrap the JSON in Markdown fences.
Do not include extra prose outside the JSON.
JSON schema:
{
  "$defs": {
    "ThemeSummary": {
      "properties": {
        "count": {
          "title": "Count",
          "type": "integer"
        },
        "example_row_ids": {
          "items": {
            "type": "integer"
          },
          "title": "Example Row Ids",
          "type": "array"
        },
        "theme": {
          "title": "Theme",
          "type": "string"
        }
      },
      "required": [
        "theme",
        "count",
        "example_row_ids"
      ],
      "title": "ThemeSummary",
      "type": "object"
    }
  },
  "properties": {
    "chunk_index": {
      "title": "Chunk Index",
      "type": "integer"
    },
    "negative_themes": {
      "items": {
        "$ref": "#/$defs/ThemeSummary"
      },
      "title": "Negative Themes",
      "type": "array"
    },
    "notable_contrasts": {
      "items": {
        "type": "string"
      },
      "title": "Notable Contrasts",
      "type": "array"
    },
    "positive_themes": {
      "items": {
        "$ref": "#/$defs/ThemeSummary"
      },
      "title": "Positive Themes",
      "type": "array"
    },
    "quality_notes": {
      "items": {
        "type": "string"
      },
      "title": "Quality Notes",
      "type": "array"
    },
    "row_count": {
      "title": "Row Count",
      "type": "integer"
    },
    "source": {
      "title": "Source",
      "type": "string"
    }
  },
  "required": [
    "source",
    "chunk_index",
    "row_count",
    "positive_themes",
    "negative_themes",
    "notable_contrasts",
    "quality_notes"
  ],
  "title": "ChunkAnalysis",
  "type": "object"
}

### Agent

_No direct message content._

<details>
<summary>Tool `finish` · Result</summary>

**Status:** Result

Return sentiment theme analysis JSON for provided dataset chunk

**Arguments**

```json
{
  "message": "{\"source\":\"amazon\",\"chunk_index\":0,\"row_count\":25,\"positive_themes\":[{\"theme\":\"sound/mic quality\",\"count\":2,\"example_row_ids\":[4,10]},{\"theme\":\"device compatibility\",\"count\":3,\"example_row_ids\":[2,7,21]},{\"theme\":\"value/quality\",\"count\":2,\"example_row_ids\":[1,13]},{\"theme\":\"battery life\",\"count\":2,\"example_row_ids\":[11,23]},{\"theme\":\"overall satisfaction\",\"count\":3,\"example_row_ids\":[15,17,18]}],\"negative_themes\":[{\"theme\":\"connection/charging issues\",\"count\":4,\"example_row_ids\":[0,3,20]},{\"theme\":\"audio/volume problems\",\"count\":2,\"example_row_ids\":[5,12]},{\"theme\":\"usability/contacts\",\"count\":1,\"example_row_ids\":[6]},{\"theme\":\"design/build comfort\",\"count\":2,\"example_row_ids\":[14,19]},{\"theme\":\"value/misleading\",\"count\":4,\"example_row_ids\":[8,9,22]}],\"notable_contrasts\":[\"Sound quality praised in positives but static/volume problems appear in negatives.\",\"Battery life is lauded in positives while charging/battery issues appear in negatives.\"],\"quality_notes\":[\"Several reviews are very short, so themes rely on sparse cues.\",\"Some positives emphasize specific device compatibility (Razr, Kindle Fire).\"]}",
  "summary": "Return sentiment theme analysis JSON for provided dataset chunk"
}
```

**Result**

```json
{"source":"amazon","chunk_index":0,"row_count":25,"positive_themes":[{"theme":"sound/mic quality","count":2,"example_row_ids":[4,10]},{"theme":"device compatibility","count":3,"example_row_ids":[2,7,21]},{"theme":"value/quality","count":2,"example_row_ids":[1,13]},{"theme":"battery life","count":2,"example_row_ids":[11,23]},{"theme":"overall satisfaction","count":3,"example_row_ids":[15,17,18]}],"negative_themes":[{"theme":"connection/charging issues","count":4,"example_row_ids":[0,3,20]},{"theme":"audio/volume problems","count":2,"example_row_ids":[5,12]},{"theme":"usability/contacts","count":1,"example_row_ids":[6]},{"theme":"design/build comfort","count":2,"example_row_ids":[14,19]},{"theme":"value/misleading","count":4,"example_row_ids":[8,9,22]}],"notable_contrasts":["Sound quality praised in positives but static/volume problems appear in negatives.","Battery life is lauded in positives while charging/battery issues appear in negatives."],"quality_notes":["Several reviews are very short, so themes rely on sparse cues.","Some positives emphasize specific device compatibility (Razr, Kindle Fire)."]}
```

</details>

In [16]:
preview_summary = preview_session.result
preview_summary.model_dump()

{'source': 'amazon',
 'chunk_index': 0,
 'row_count': 100,
 'positive_themes': [{'theme': 'features/usability',
   'count': 30,
   'example_row_ids': [2, 34, 78]},
  {'theme': 'durability/build', 'count': 9, 'example_row_ids': [13, 57, 91]},
  {'theme': 'audio quality', 'count': 6, 'example_row_ids': [4, 10, 58]},
  {'theme': 'battery life', 'count': 2, 'example_row_ids': [11, 23]},
  {'theme': 'value/price', 'count': 4, 'example_row_ids': [1, 42, 88]}],
 'negative_themes': [{'theme': 'value/dissatisfaction',
   'count': 15,
   'example_row_ids': [8, 9, 94]},
  {'theme': 'usability/compatibility',
   'count': 13,
   'example_row_ids': [0, 20, 76]},
  {'theme': 'audio/mic quality', 'count': 8, 'example_row_ids': [12, 28, 93]},
  {'theme': 'battery life', 'count': 7, 'example_row_ids': [3, 29, 83]},
  {'theme': 'durability/defects',
   'count': 6,
   'example_row_ids': [30, 66, 84]}],
 'notable_contrasts': ['Audio quality is praised in several positives but criticized for static, low vol

## Step 5 - Run the full batch in parallel

This is the map stage. The code owns chunk scheduling and failure handling; each successful session already carries a validated `ChunkAnalysis` result.


In [24]:
parallel_results = agent.parallel(
    chunks,
    build_chunk_request,
    max_concurrency=MAX_CONCURRENCY,
)

failures = [result for result in parallel_results if isinstance(result, ParallelFailure)]
sessions = [result for result in parallel_results if not isinstance(result, ParallelFailure)]

{
    "requested_chunks": len(chunks),
    "completed_sessions": len(sessions),
    "failures": len(failures),
}


{'requested_chunks': 30, 'completed_sessions': 29, 'failures': 1}

In [26]:
chunk_summaries = [session.result for session in sessions]

if not chunk_summaries:
    raise RuntimeError("No successful chunk summaries were produced.")

failure_summary = pd.DataFrame(
    [
        {
            "index": failure.index,
            "source": failure.item.source,
            "chunk_index": failure.item.chunk_index,
            "phase": failure.phase,
            "error": str(failure.error),
        }
        for failure in failures
    ]
)

chunk_summary_overview = pd.DataFrame(
    [
        {
            "source": summary.source,
            "chunk_index": summary.chunk_index,
            "row_count": summary.row_count,
            "positive_theme_count": len(summary.positive_themes),
            "negative_theme_count": len(summary.negative_themes),
        }
        for summary in chunk_summaries
    ]
)

failure_summary if not failure_summary.empty else chunk_summary_overview.head(10)


,index,source,chunk_index,phase,error
0,1,amazon,1,run,Conversation run failed for id=a1637775-c9b1-45e8-b5cc-69c3bb748a83: [Errno 9] Bad file descriptor


## Step 6 - Reduce the chunk outputs with plain Python

This is the reduce stage. The agent returns typed chunk summaries; the merge step is ordinary `pandas` code.


In [27]:
def normalize_theme(theme: str) -> str:
    return " ".join(theme.strip().lower().split())



def collect_theme_rows(chunk_summaries: list[ChunkAnalysis]) -> pd.DataFrame:
    rows: list[dict[str, str | int]] = []
    for summary in chunk_summaries:
        for themes, sentiment in (
            (summary.positive_themes, "positive"),
            (summary.negative_themes, "negative"),
        ):
            for theme in themes:
                rows.append(
                    {
                        "source": summary.source,
                        "chunk_index": summary.chunk_index,
                        "sentiment": sentiment,
                        "theme": normalize_theme(theme.theme),
                        "count": theme.count,
                        "example_row_ids": ", ".join(
                            str(row_id) for row_id in theme.example_row_ids[:3]
                        ),
                    }
                )
    return pd.DataFrame(rows)



theme_rows = collect_theme_rows(chunk_summaries)
if theme_rows.empty:
    raise RuntimeError("Agent returned no theme rows to aggregate.")

theme_rows.head(10)


,source,chunk_index,sentiment,theme,count,example_row_ids
0,amazon,0,positive,device_compatibility,4,"2, 7, 21"
1,amazon,0,positive,overall_satisfaction,2,"17, 18"
2,amazon,0,positive,audio_quality,2,"4, 10"
3,amazon,0,positive,battery_life,2,"11, 23"
4,amazon,0,positive,build_value,2,"1, 13"
5,amazon,0,negative,connection_compatibility,4,"0, 12, 20"
6,amazon,0,negative,power_battery,2,"3, 24"
7,amazon,0,negative,comfort_durability,2,"14, 19"
8,amazon,0,negative,usability_limitations,1,6
9,amazon,0,negative,waste_misleading,4,"8, 9, 22"


In [28]:
top_themes_by_source = (
    theme_rows.groupby(["source", "sentiment", "theme"], as_index=False)["count"]
    .sum()
    .sort_values(["source", "sentiment", "count", "theme"], ascending=[True, True, False, True])
)

top_themes_by_source.groupby(["source", "sentiment"]).head(5).reset_index(drop=True)


,source,sentiment,theme,count
0,amazon,negative,dissatisfaction or waste,6
1,amazon,negative,overall_dissatisfaction_value,6
2,amazon,negative,audio_mic_issues,4
3,amazon,negative,connection_compatibility,4
4,amazon,negative,defective or failed,4
5,amazon,positive,overall satisfaction,8
6,amazon,positive,overall_satisfaction,7
7,amazon,positive,overall_satisfaction_value,7
8,amazon,positive,performance & features,5
9,amazon,positive,performance/works,5


In [29]:
overall_top_themes = (
    theme_rows.groupby(["sentiment", "theme"], as_index=False)["count"]
    .sum()
    .sort_values(["sentiment", "count", "theme"], ascending=[True, False, True])
)

overall_top_themes.groupby("sentiment").head(10).reset_index(drop=True)


,sentiment,theme,count
0,negative,overall bad,11
1,negative,poor food quality,10
2,negative,overall dissatisfaction,9
3,negative,terrible quality,9
4,negative,bad food quality,7
5,negative,dissatisfaction or waste,6
6,negative,food/drink quality or value,6
7,negative,overall awful,6
8,negative,overall_dissatisfaction_value,6
9,negative,awkward tone/pretentious/politically correct scenes,5


## Step 7 - Run one reducer pass over the merged tables

The reducer agent sees aggregated data instead of raw rows. This is the second agent stage in the same overall workflow shape.


In [30]:
top_reducer_table = (
    top_themes_by_source.groupby(["source", "sentiment"])
    .head(5)
    .reset_index(drop=True)
)

reducer_request = f"""
Write a short markdown summary of the aggregated chunk analysis below.

Use this structure:
- One headline sentence.
- Three bullets comparing amazon, imdb, and yelp.
- One bullet about the limitations of chunked agent analysis.
- One bullet describing the next improvement to try.

Use only the aggregated data below.

{top_reducer_table.to_string(index=False)}
""".strip()

reducer_session = reducer_request >> agent
reducer_session


Status: **Finished**

2 turns · 0 tool calls

### User

1. Write a short markdown summary of the aggregated chunk analysis below.
   
   Use this structure:
   - One headline sentence.
   - Three bullets comparing amazon, imdb, and yelp.
   - One bullet about the limitations of chunked agent analysis.
   - One bullet describing the next improvement to try.
   
   Use only the aggregated data below.
   
   source sentiment                                               theme  count
   amazon  negative                            dissatisfaction or waste      6
   amazon  negative                       overall_dissatisfaction_value      6
   amazon  negative                                    audio_mic_issues      4
   amazon  negative                            connection_compatibility      4
   amazon  negative                                 defective or failed      4
   amazon  positive                                overall satisfaction      8
   amazon  positive                                overall_satisfaction      7
   amazon  positive                          overall_satisfaction_value      7
   amazon  positive                              performance & features      5
   amazon  positive                                   performance/works      5
     imdb  negative                                         overall bad     11
     imdb  negative                                    terrible quality      9
     imdb  negative                                       overall awful      6
     imdb  negative awkward tone/pretentious/politically correct scenes      5
     imdb  negative                                bad writing and plot      5
     imdb  positive                               recommendation/praise      9
     imdb  positive                                 enthusiastic_praise      7
     imdb  positive                                   actor_performance      6
     imdb  positive                       charming elements and setting      6
     imdb  positive                       general praise/recommendation      6
     yelp  negative                                   poor food quality     10
     yelp  negative                                    bad food quality      7
     yelp  negative                         food/drink quality or value      6
     yelp  negative                                    bad_food_quality      5
     yelp  negative                                food quality/texture      5
     yelp  positive                                        food quality     26
     yelp  positive                                        food_quality     14
     yelp  positive                                          tasty food     11
     yelp  positive                                    friendly service      9
     yelp  positive                                          tasty_food      8

### Agent

Amazon reviews mix strong satisfaction themes with recurring mic/connection defects, IMDB skews more negative on overall quality despite some praise, and Yelp is dominated by food quality sentiment.

- **Amazon vs IMDB vs Yelp:** Amazon shows more balanced sentiment with positives on overall satisfaction/performance but notable defects/compatibility issues, IMDB leans negative on overall quality and writing despite actor praise, and Yelp is polarized around food quality with strong positives (tasty food) and substantial negatives (poor/bad food quality).
- **Amazon vs IMDB vs Yelp:** Amazon positives emphasize performance/works and value satisfaction, IMDB positives focus on recommendations, acting, and charm, while Yelp positives highlight food quality and friendly service.
- **Amazon vs IMDB vs Yelp:** Amazon negatives cluster around hardware issues, IMDB negatives around overall bad/terrible quality and plot, and Yelp negatives around food quality/value/texture.

- **Limitation:** Chunked agent analysis may overemphasize repeated theme labels and miss nuanced context across sources.
- **Next improvement:** Normalize or merge near-duplicate theme labels (e.g., “food quality” vs “food_quality”) before summarizing.

## Exercises

- Change `CHUNK_SIZE` from `25` to `10` and compare whether the dominant themes stay stable.
- Rerun only one source, such as `amazon`, to measure cost and latency tradeoffs.
- Replace `build_chunk_request(...)` with a pure Python keyword extractor and keep the same split/map/reduce notebook shape.

Common pitfall:
- If `session.result` raises a validation error, inspect `session.result_text` and tighten the request instructions.

Extension:
- Save `[summary.model_dump() for summary in chunk_summaries]` to disk and compare multiple model runs against the same merged reducer stage.


In [ ]:
# Exercise scaffold: rerun a smaller slice with a smaller chunk size.
smaller_chunks = [chunk for chunk in build_chunks(dataset, 10) if chunk.source == "amazon"][:2]
smaller_chunks[0].rows.head(5)
